# Exploratory Data Analysis (EDA)

This notebook inspects the reviews dataset and starts with loading data plus checking available columns.

In [1]:
import pandas as pd
from pathlib import Path

# STEP 6 - Load the dataset
# If CSV:
# df = pd.read_csv("../data/raw/reviews.csv")

# If JSON (lines=True):
# df = pd.read_json("../data/raw/reviews.json", lines=True)

raw_dir = Path("../data/raw")
if not raw_dir.exists():
    raw_dir = Path("data/raw")

csv_path = raw_dir / "reviews.csv"
json_path = raw_dir / "reviews.json"
jsonl_gz_files = sorted(raw_dir.glob("*.jsonl.gz"))

if csv_path.exists():
    df = pd.read_csv(csv_path)
elif json_path.exists():
    df = pd.read_json(json_path, lines=True)
elif jsonl_gz_files:
    # Fallback for compressed JSON Lines files already present in this project
    df = pd.read_json(jsonl_gz_files[0], lines=True, compression="infer")
else:
    raise FileNotFoundError("No reviews.csv, reviews.json, or .jsonl.gz file found in data/raw")

df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Cell Phones & Accessories,ARAREE Slim Diary Cell Phone Case for Samsung ...,3.8,5,"[Genuine Cow leather with 6 different colors, ...","[JUST LOOK, You can tell the difference. Make ...",None,[{'thumb': 'https://m.media-amazon.com/images/...,[],araree,"[Cell Phones & Accessories, Cases, Holsters & ...",{'Product Dimensions': '3.35 x 0.59 x 6.18 inc...,B013SK1JTY,NaN,NaN,NaN
1,Cell Phones & Accessories,Bastmei for OnePlus 7T Case Extremely Light Ul...,4.4,177,[Ultra-thin & Ultra-light: The ultra slim fit ...,[],11.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Bastmei,"[Cell Phones & Accessories, Cases, Holsters & ...",{'Package Dimensions': '7.6 x 4.29 x 0.75 inch...,B07ZPSG8P5,NaN,NaN,NaN
2,Cell Phones & Accessories,Wireless Fones Branded New Iphone 5C/LITE Hot ...,4.0,2,[],[],None,[{'thumb': 'https://m.media-amazon.com/images/...,[],WIRELESS FONES,"[Cell Phones & Accessories, iPhone Accessories]","{'Item model number': 'Apple Iphone 5C', 'Othe...",B00GKR3L12,NaN,NaN,NaN
3,Cell Phones & Accessories,"iPhone 6 Plus + Case, DandyCase Perfect PATTER...",4.0,15,"[Slim-Fit design for the iPhone 6 Plus (5.5"" s...",[Case does not need to be removed for charging...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],DandyCase,"[Cell Phones & Accessories, iPhone Accessories]",{'Product Dimensions': '5.43 x 0.28 x 2.64 inc...,B00PB8U8BW,NaN,NaN,NaN
4,Cell Phones & Accessories,"Case for Galaxy S6/S6 Edge, Thin Translucent V...",4.0,1,[],[],None,[{'thumb': 'https://m.media-amazon.com/images/...,[],7Pite,"[Cell Phones & Accessories, Cases, Holsters & ...",{'Package Dimensions': '8.31 x 3.74 x 0.55 inc...,B07D3RHSRV,NaN,NaN,NaN


In [2]:
# STEP 7 - Inspect columns

df.columns

Index(['main_category', 'title', 'average_rating', 'rating_number', 'features',
       'description', 'price', 'images', 'videos', 'store', 'categories',
       'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
      dtype='str')

In [2]:
# STEP 8 - Reduce dataset size

sample_size = min(5000, len(df))
df = df.sample(sample_size, random_state=42)

# STEP 9 - Clean the dataset
# Preserve real user identity by mapping canonical fields from common alternatives.

def first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

user_col = first_existing(
    df.columns,
    [
        "user_id",
        "reviewerID",
        "reviewer_id",
        "user",
        "uid",
        "customer_id",
        "profile_id",
        "author",
    ],
)
product_col = first_existing(df.columns, ["product_id", "asin", "parent_asin", "item_id"])
rating_col = first_existing(df.columns, ["rating", "overall", "stars", "average_rating"])
text_col = first_existing(df.columns, ["review_text", "review_body", "text", "review", "title"])

if user_col and user_col != "user_id":
    df["user_id"] = df[user_col]
if product_col and product_col != "product_id":
    df["product_id"] = df[product_col]
if rating_col and rating_col != "rating":
    df["rating"] = df[rating_col]
if text_col and text_col != "review_text":
    df["review_text"] = df[text_col]

# Fallback: if no user-like field exists, use store as a stable identity proxy.
if "user_id" not in df.columns and "store" in df.columns:
    df["user_id"] = df["store"]
    print("Warning: using store as a proxy for user_id because review-level user IDs were not found.")

required = ["user_id", "product_id", "rating", "review_text"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(
        f"Missing required columns for user profiling: {missing_required}. "
        "Load a review-level dataset that includes user identifiers (for example reviewerID/user_id)."
    )

# Keep important columns and drop missing rows.
df = df[required].dropna(subset=required)

# Ensure user IDs are not blank placeholders.
df["user_id"] = df["user_id"].astype(str).str.strip()
df = df[df["user_id"] != ""]

print(f"Rows after cleaning: {len(df):,}")
print(f"Unique users preserved: {df['user_id'].nunique():,}")
df.info()

ValueError: Missing required columns for user profiling: ['user_id']. Load a review-level dataset that includes user identifiers (for example reviewerID/user_id).

In [5]:
# STEP 10 - Save clean dataset

from pathlib import Path

processed_dir = Path("../data/processed")
if not processed_dir.exists():
    processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "clean_reviews.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df):,} rows to {output_path.resolve()}")

Saved 5,000 rows to C:\Users\danie\bct-agent\data\processed\clean_reviews.csv


In [9]:
# STEP 11 - Create user profiles (persona memory)

user_profiles = df.groupby("user_id").agg({
    "rating": ["mean", "count"],
    "review_text": lambda x: " ".join(x.head(3))
})

# Flatten multi-level columns created by agg.
user_profiles.columns = [
    "avg_rating",
    "review_count",
    "sample_reviews"
]

user_profiles = user_profiles.reset_index()

user_profiles.head()

,user_id,avg_rating,review_count,sample_reviews
0,unknown_user,3.9445,5000,Tksafy Case for Samsung Galaxy S21 FE 5G Case ...


In [3]:
# STEP 17 - Create product profiles (item memory)

item_profiles = df.groupby("product_id").agg({
    "rating": ["mean", "count"],
    "review_text": lambda x: " ".join(x.head(3))
})

item_profiles.columns = [
    "avg_rating",
    "review_count",
    "sample_reviews"
]

item_profiles = item_profiles.reset_index()
item_profiles.head()

,product_id,avg_rating,review_count,sample_reviews
0,099903961X,5.0,1,The Story of Tee-Blanc Sambeaux (Little White ...
1,B00006RZ51,3.0,1,"TracFone Prepaid Airtime Card, 150 Minutes for..."
2,B0001TDH50,4.0,1,Nokia 3595 Phone (AT&T)
3,B0006H4FVC,3.5,1,Wilson Electronics Dual Band - 800-1900 MHz Lo...
4,B0006JI3K4,3.7,1,Palm Treo 600/650 Premium Side Case


In [ ]:
# STEPS 18-19 - Better user personas + natural language persona text

df["review_length"] = df["review_text"].astype(str).apply(len)

user_profiles = df.groupby("user_id").agg({
    "rating": ["mean", "count"],
    "review_length": "mean",
    "review_text": lambda x: " ".join(x.head(5))
})

user_profiles.columns = [
    "avg_rating",
    "review_count",
    "avg_review_length",
    "sample_reviews"
]

user_profiles = user_profiles.reset_index()


def build_persona(row):
    if row["avg_rating"] >= 4:
        mood = "generally positive"
    elif row["avg_rating"] >= 3:
        mood = "balanced"
    else:
        mood = "critical"

    if row["avg_review_length"] > 300:
        style = "detailed reviewer"
    else:
        style = "short-form reviewer"

    return (
        f"User is a {mood} customer and a {style}.\n"
        f"Average rating given: {row['avg_rating']:.1f}\n"
        f"Total reviews: {int(row['review_count'])}\n"
        f"Example behavior:\n{str(row['sample_reviews'])[:300]}"
    )


user_profiles["persona"] = user_profiles.apply(build_persona, axis=1)

print(user_profiles["persona"].iloc[0] if not user_profiles.empty else "No user profiles generated")
user_profiles.head()

In [ ]:
# STEP 20 - Build recommendation memory from liked items

liked_items = df[df["rating"] >= 4]

user_likes = liked_items.groupby("user_id")["product_id"].apply(list)
user_likes = user_likes.reset_index(name="liked_products")

user_likes.head()

In [ ]:
# STEPS 21-24 - Install model deps, create embeddings, and semantic recommender

# If needed in a fresh environment, run this once:
# %pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

item_profiles["embedding"] = item_profiles["sample_reviews"].apply(
    lambda x: model.encode(str(x))
)


def recommend_similar_products(product_id, top_k=5):
    target = item_profiles[item_profiles["product_id"] == product_id]
    if target.empty:
        return None

    target_embedding = np.array(target.iloc[0]["embedding"]).reshape(1, -1)
    similarities = []

    for _, row in item_profiles.iterrows():
        emb = np.array(row["embedding"]).reshape(1, -1)
        sim = cosine_similarity(target_embedding, emb)[0][0]
        similarities.append(sim)

    ranked = item_profiles.copy()
    ranked["similarity"] = similarities

    recommendations = ranked.sort_values("similarity", ascending=False).head(top_k)
    return recommendations[["product_id", "avg_rating", "similarity"]]


if not item_profiles.empty:
    recommend_similar_products(item_profiles["product_id"].iloc[0])
else:
    print("No item profiles available")

In [ ]:
# STEP 27 - Save user and item profiles

from pathlib import Path

processed_dir = Path("../data/processed")
if not processed_dir.exists():
    processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

user_profiles_path = processed_dir / "user_profiles.csv"
item_profiles_path = processed_dir / "item_profiles.csv"

user_profiles.to_csv(user_profiles_path, index=False)
item_profiles.drop(columns=["embedding"], errors="ignore").to_csv(item_profiles_path, index=False)

print(f"Saved user profiles: {user_profiles_path.resolve()}")
print(f"Saved item profiles: {item_profiles_path.resolve()}")